# In this file, we test the trained unet model on the 2p_5p NFI Dataset. Then we compare the metrics of the model compared to human specialists.

In [16]:
%cd /Users/amarmesic/Documents/tudelft/thesis/DNANet

/Users/amarmesic/Documents/tudelft/thesis/DNANet


/Users/amarmesic/miniconda3/envs/dnanet/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [17]:
from DNAnet.data.data_models.hid_dataset import HIDDataset
from DNAnet.evaluation import *
from DNAnet.evaluation.segmentation.allele_metrics import *
from DNAnet.evaluation.segmentation.pixel_metrics import *

from config_io import load_model, load_dataset

In [18]:
dataset = load_dataset("provedit_data.yaml")

2025-10-11 14:00:53 INFO     Found 675 files in /Users/amarmesic/Documents/tudelft/thesis/datasets/USE THIS - PROVEDIt_2-5-Person Profiles_3500 5sec_GF29cycles
2025-10-11 14:00:54 WARNING  Size standard for E04_RD14-0003-33_34-1,2-M4d-0.045GF-Q1.3_05.5sec.hid differs 18.912544484957948 from the expected 
2025-10-11 14:00:56 WARNING  Size standard for H08_RD14-0003-49_50_29-1,4,1-M2e-0.378GF-Q1.4_08.5sec.hid differs 13.401428160223645 from the expected 
2025-10-11 14:00:57 WARNING  Size standard for G03_RD14-0003-40_41-1,4-M3e-0.625GF-Q0.8_07.5sec.hid differs 12.982847650856456 from the expected 
2025-10-11 14:01:00 WARNING  Size standard for E03_RD14-0003-48_49_50_29-1,4,4,4-M2I15-0.75GF-Q1.1_05.5sec.hid differs 13.001695302289932 from the expected 
2025-10-11 14:01:01 WARNING  Size standard for E05_RD14-0003-46_47_48-1,1,1-M3I35-0.189GF-QLAND_05.5sec.hid differs 18.824515396807158 from the expected 
2025-10-11 14:01:05 INFO     Number of valid images: 670
2025-10-11 14:01:05 INFO     

In [19]:
# my_unet_model = load_model("config/models/unet_advanced.yaml")
# my_unet_model.load("output/ProvedIt_best_unet/20250926_142826/checkpoint/")

dnanet_best_unet_model = load_model("config/models/unet.yaml")
dnanet_best_unet_model.load("resources/model/current_best_unet/")

2025-10-11 14:01:31 INFO     Using device: cpu


In [20]:
# my_predictions = my_unet_model.predict_batch(dataset)
predictions = dnanet_best_unet_model.predict_batch(dataset)

2025-10-11 15:24:19 INFO     Calling alleles from predicted segmentation...
2025-10-11 15:24:20 WARNING  No predictions present in dye row 3
2025-10-11 15:24:20 WARNING  No predictions present in dye row 3
2025-10-11 15:24:20 WARNING  No predictions present in dye row 0
2025-10-11 15:24:20 WARNING  No predictions present in dye row 3
2025-10-11 15:24:22 WARNING  No predictions present in dye row 0
2025-10-11 15:24:22 WARNING  No predictions present in dye row 2
2025-10-11 15:24:22 WARNING  No predictions present in dye row 3
2025-10-11 15:24:22 WARNING  No predictions present in dye row 3
2025-10-11 15:24:23 WARNING  No predictions present in dye row 3
2025-10-11 15:24:23 WARNING  No predictions present in dye row 3


## 30 Mixtures:
### Mine:
allele u-net f1:  0.9269633957724694
allele u-net precision:  0.9387399930386355
allele u-net recall:  0.9154786150712831

### Original:
allele u-net f1:  0.9696233292831107
allele u-net precision:  0.9921847246891652
allele u-net recall:  0.9480651731160896

In [21]:
from DNAnet.evaluation.segmentation.allele_metrics import allele_f1_score, allele_precision, allele_recall
from DNAnet.evaluation.segmentation.pixel_metrics import pixel_f1_score, pixel_precision, pixel_recall

f1 = pixel_f1_score(dataset, predictions)
precision = pixel_precision(dataset, predictions)
recall = pixel_recall(dataset, predictions)
print(f"- Pixel F1 Score: {f1:.4g}")
print(f"- Pixel Precision: {precision:.4g}")
print(f"- Pixel Recall: {recall:.4g}")

alelle_f1 = allele_f1_score(dataset, predictions)
print(f'- Allele F1 Score: {alelle_f1:.4g}')
print(f'- Allele Precision: {allele_precision(dataset, predictions):.4g}')
print(f'- Allele Recall: {allele_recall(dataset, predictions):.4g}')

- Pixel F1 Score: 0.8073
- Pixel Precision: 0.8896
- Pixel Recall: 0.739
- Allele F1 Score: 0.7747
- Allele Precision: 0.9222
- Allele Recall: 0.6678


In [22]:
import neptune
run = neptune.init_run(
    name="本地-最好U网-考试整ProvedIt-预：缩放",
    project="amar-mesic/dna-thesis",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJkOTQ1Njc4MC0yOTcyLTRlMmQtYTMwMy0xOGYxZTAwMmIzZGUifQ==",
)

run["test/pixel_f1"] = float(f"{f1:.4g}")
run["test/pixel_precision"] = float(f"{precision:.4g}")
run["test/pixel_recall"] = float(f"{recall:.4g}")
run["test/allele_f1"] = float(f"{alelle_f1:.4g}")
run["test/allele_precision"] = float(f"{allele_precision(dataset, predictions):.4g}")
run["test/allele_recall"] = float(f"{allele_recall(dataset, predictions):.4g}")

meta = {
    "experiment": "DNANet_Baseline",
    "model": "DNANet_Pretrained",
    "dataset": "proved_it",
}
for k, v in meta.items():
    run[f'meta/{k}'] = v
run.stop()

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-793
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 9 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 9 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-793/metadata
